# Lecture 3.6 — Hands-On: Build a Codebase TODO Summarizer Agent

This is the payoff lecture for Section 3. We bring every built-in tool we learned in
Lectures 3.1–3.5 together into one complete, useful agent.

**What we are building:**
A TODO Summarizer Agent that:
1. Asks the user how they want the report formatted (`AskUserQuestion`)
2. Finds all files in the project (`Glob`)
3. Searches for all TODO comments (`Grep`)
4. Reads context around each TODO when requested (`Read`)
5. Produces a clean, organised summary report

**Tools used:** `Glob`, `Grep`, `Read`, `AskUserQuestion`

In [1]:
# Install the Claude Agent SDK

# We pin the Claude Agent SDK to a specific version to ensure all examples
# in this notebook run exactly as shown in the course. If you encounter any
# issues or want to experiment with newer features, you can install the latest
# version by removing the version pin (replace 'claude-agent-sdk==0.2.93' with just
# 'claude-agent-sdk'). Note that newer versions may behave differently from
# what is demonstrated in the videos. We will update the notebooks periodically
# to keep up with new releases.

!pip install claude-agent-sdk==0.2.93 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 MB 11.7 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Model Configuration

We use `claude-haiku-4-5` by default — a fast, cost-effective model that works well for
agentic tasks like this one. You can change this to any model you have access to.

For the latest available models, see:
https://platform.claude.com/docs/en/about-claude/models/overview

In [3]:
# Model configuration
# Change this to use a different Claude model
# For the latest available models visit:
# https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-4-5"

## The Sample Project

We create a realistic Python project programmatically — 8 files with 9 TODO comments
planted deliberately across `src/`, `tests/`, and `config/`.

Why do it this way? Because every student gets the exact same files, the same TODOs,
and the same expected output. No surprises.

```
/content/todo_project/
├── README.md
├── src/
│   ├── auth.py              # 2 TODOs
│   ├── payments.py          # 2 TODOs
│   ├── notifications.py     # 1 TODO
│   └── utils.py             # 1 TODO
├── tests/
│   ├── test_auth.py         # 1 TODO
│   └── test_payments.py     # 1 TODO
└── config/
    └── settings.py          # 1 TODO
```

In [4]:
import os

os.makedirs("/content/todo_project/src", exist_ok=True)
os.makedirs("/content/todo_project/tests", exist_ok=True)
os.makedirs("/content/todo_project/config", exist_ok=True)

with open("/content/todo_project/README.md", "w") as f:
    f.write("# TODO Project\n")
    f.write("A demo project for the TODO Summarizer Agent.\n")
    f.write("This project has authentication, payments, notifications, and utility modules.\n")

with open("/content/todo_project/src/auth.py", "w") as f:
    f.write("# Authentication module\n")
    f.write("# TODO: Add password hashing before storing credentials\n")
    f.write("# TODO: Implement OAuth2 support\n")
    f.write("def login(user, password):\n")
    f.write("    pass\n\n")
    f.write("def logout(user):\n")
    f.write("    pass\n\n")
    f.write("def reset_password(user):\n")
    f.write("    pass\n")

with open("/content/todo_project/src/payments.py", "w") as f:
    f.write("# Payments module\n")
    f.write("# TODO: Integrate Stripe payment gateway\n")
    f.write("# TODO: Add retry logic for failed transactions\n")
    f.write("def process_payment(amount, card):\n")
    f.write("    pass\n\n")
    f.write("def refund_payment(transaction_id):\n")
    f.write("    pass\n")

with open("/content/todo_project/src/notifications.py", "w") as f:
    f.write("# Notifications module\n")
    f.write("# TODO: Add email template support\n")
    f.write("def send_email(to, subject, body):\n")
    f.write("    pass\n\n")
    f.write("def send_sms(to, message):\n")
    f.write("    pass\n")

with open("/content/todo_project/src/utils.py", "w") as f:
    f.write("# Utility functions\n")
    f.write("# TODO: Add input validation for all utility functions\n")
    f.write("def format_date(date):\n")
    f.write("    pass\n\n")
    f.write("def format_currency(amount):\n")
    f.write("    pass\n")

with open("/content/todo_project/tests/test_auth.py", "w") as f:
    f.write("# Tests for auth module\n")
    f.write("# TODO: Add edge case tests for invalid credentials\n")
    f.write("def test_login():\n")
    f.write("    pass\n\n")
    f.write("def test_logout():\n")
    f.write("    pass\n")

with open("/content/todo_project/tests/test_payments.py", "w") as f:
    f.write("# Tests for payments module\n")
    f.write("# TODO: Add mock tests for payment gateway\n")
    f.write("def test_process_payment():\n")
    f.write("    pass\n\n")
    f.write("def test_refund():\n")
    f.write("    pass\n")

with open("/content/todo_project/config/settings.py", "w") as f:
    f.write("# Application settings\n")
    f.write("# TODO: Move sensitive settings to environment variables\n")
    f.write("DEBUG = True\n")
    f.write("DATABASE_URL = 'sqlite:///db.sqlite3'\n")
    f.write("SECRET_KEY = 'dev-secret-key'\n")

print("Sample project created.")

Sample project created.


## Verifying What Was Created

Before we run the agent, let's see exactly what files exist and what TODOs are in them.
This gives us a ground-truth reference — when the agent produces its report, we can
verify the results against what we see here.

In [5]:
import os

# Show directory structure
print("Project structure:")
print("=" * 40)
for root, dirs, files in os.walk("/content/todo_project"):
    level = root.replace("/content/todo_project", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

# Show all file contents
files_to_show = [
    "/content/todo_project/README.md",
    "/content/todo_project/src/auth.py",
    "/content/todo_project/src/payments.py",
    "/content/todo_project/src/notifications.py",
    "/content/todo_project/src/utils.py",
    "/content/todo_project/tests/test_auth.py",
    "/content/todo_project/tests/test_payments.py",
    "/content/todo_project/config/settings.py",
]

print("\nFile contents:")
print("=" * 40)
for filepath in files_to_show:
    print(f"\n--- {filepath} ---")
    with open(filepath, "r") as f:
        print(f.read())

Project structure:
todo_project/
  README.md
  config/
    settings.py
  src/
    notifications.py
    auth.py
    payments.py
    utils.py
  tests/
    test_payments.py
    test_auth.py

File contents:

--- /content/todo_project/README.md ---
# TODO Project
A demo project for the TODO Summarizer Agent.
This project has authentication, payments, notifications, and utility modules.


--- /content/todo_project/src/auth.py ---
# Authentication module
# TODO: Add password hashing before storing credentials
# TODO: Implement OAuth2 support
def login(user, password):
    pass

def logout(user):
    pass

def reset_password(user):
    pass


--- /content/todo_project/src/payments.py ---
# Payments module
# TODO: Integrate Stripe payment gateway
# TODO: Add retry logic for failed transactions
def process_payment(amount, card):
    pass

def refund_payment(transaction_id):
    pass


--- /content/todo_project/src/notifications.py ---
# Notifications module
# TODO: Add email template support
def

## Planning the Agent — Design Before Code

Before writing any agent code, let's map out what the agent needs to do.

**The TODO Summarizer Agent — step by step:**

| Step | Tool | What it does |
|------|------|--------------|
| 1 | `AskUserQuestion` | Ask the user how to format the report and which TODOs to include |
| 2 | `Glob` | Find all files in `/content/todo_project` |
| 3 | `Grep` | Search for `TODO` comments across those files |
| 4 | `Read` | Get surrounding code context (only if Detailed format chosen) |
| 5 | — | Produce a clean report based on the user's preferences |

**Pattern needed from Lecture 3.5:**
- Streaming input (`prompt_stream` async generator)
- Dummy `PreToolUse` hook (keeps the HTTP connection alive)
- `can_use_tool` callback (intercepts `AskUserQuestion`)

## The AskUserQuestion Handler

This is the same `AskUserQuestion` handler we built in Lecture 3.5 — same streaming
input pattern, same dummy hook, same `can_use_tool` callback. We bring it straight
into this agent.

In [6]:
from claude_agent_sdk.types import HookMatcher, PermissionResultAllow


def parse_response(response: str, options: list) -> str:
    try:
        indices = [int(s.strip()) - 1 for s in response.split(",")]
        labels = [options[i]["label"] for i in indices if 0 <= i < len(options)]
        return ", ".join(labels) if labels else response
    except ValueError:
        return response


async def handle_ask_user_question(input_data: dict):
    answers = {}

    for q in input_data.get("questions", []):
        print(f"\n{q['header']}: {q['question']}")

        options = q["options"]
        for i, opt in enumerate(options):
            print(f"  {i + 1}. {opt['label']} - {opt['description']}")
        if q.get("multiSelect"):
            print("  (Enter numbers separated by commas, or type your own answer)")
        else:
            print("  (Enter a number, or type your own answer)")

        response = input("Your choice: ").strip()
        answers[q["question"]] = parse_response(response, options)

    return PermissionResultAllow(
        updated_input={
            "questions": input_data.get("questions", []),
            "answers": answers,
        }
    )


async def can_use_tool(tool_name: str, input_data: dict, context):
    if tool_name == "AskUserQuestion":
        return await handle_ask_user_question(input_data)
    return PermissionResultAllow(updated_input=input_data)


async def dummy_hook(input_data, tool_use_id, context):
    return {"continue_": True}


print("Handler functions defined.")

Handler functions defined.


## The TODO Summarizer Agent — The Prompt

The `prompt_stream` async generator delivers our instructions to the agent.
The prompt describes the agent's role, the two questions to ask, the tools to use,
and how to format the final report — but not the exact order to call the tools.
The agent reasons about that itself.

In [7]:
async def prompt_stream():
    yield {
        "type": "user",
        "message": {
            "role": "user",
            "content": """
            You are a TODO Summarizer Agent. Your job is to find and
            summarise all TODO comments in the project at /content/todo_project.

            Before starting, ask the user two clarifying questions:
            You MUST use the AskUserQuestion tool to ask me clarifying questions.
            Do NOT write questions as plain text — only use the AskUserQuestion tool.

            Question 1 — Report format:
            - Brief: just the TODO text and file name
            - Detailed: TODO text, file name, and surrounding code context

            Question 2 — Filter by priority:
            - All TODOs: show everything found
            - High priority only: show only TODOs related to security,
              authentication, or payments
            - By location: show only TODOs in src/ files

            Then:
            1. Use Glob to find all files in /content/todo_project
            2. Use Grep to find all TODO comments
            3. Use Read to get context around each TODO if Detailed format
               was chosen
            4. Produce a clean organised report based on the user's
               format and filter preferences

            Present the final report with file names as headers and
            TODOs listed under each file.
            """,
        },
    }

## Running the Agent

When you run the next cell, the agent will pause twice and ask you two questions.
Answer each one by typing a number and pressing Enter.
After both answers, the agent will continue automatically.

**Expected output — Brief + All TODOs:**
```
TODO Summary Report
===================
src/auth.py
  - TODO: Add password hashing before storing credentials
  - TODO: Implement OAuth2 support
...
Total: 9 TODOs found across 8 files
```

**Expected output — Detailed + High priority only:**
```
TODO Summary Report — High Priority
=====================================
src/auth.py
  TODO: Add password hashing before storing credentials
  Context:
    # Authentication module
    # TODO: Add password hashing before storing credentials
    def login(user, password):
...
Total: 4 high priority TODOs found across 2 files
```

In [8]:
from claude_agent_sdk import ClaudeAgentOptions, ResultMessage, query

async for message in query(
    prompt=prompt_stream(),
    options=ClaudeAgentOptions(
        allowed_tools=["Glob", "Grep", "Read", "AskUserQuestion"],
        can_use_tool=can_use_tool,
        model=MODEL_NAME,
        hooks={
            "PreToolUse": [
                HookMatcher(matcher=None, hooks=[dummy_hook])
            ]
        },
    ),
):
    if isinstance(message, ResultMessage) and message.subtype == "success":
        print(message.result)


Report Format: What report format would you prefer?
  1. Brief - Show just the TODO text and file name
  2. Detailed - Show TODO text, file name, and surrounding code context
  (Enter a number, or type your own answer)
Your choice: 2

Filter Preference: How would you like to filter the TODOs?
  1. All TODOs - Show everything found in the project
  2. High priority only - Show only TODOs related to security, authentication, or payments
  3. By location - Show only TODOs in src/ files
  (Enter a number, or type your own answer)
Your choice: 1
Perfect! Now let me compile a comprehensive detailed report of all TODOs found in the project.

---

# TODO Summarizer Report

## Summary
**Total TODOs Found:** 10 across 7 files

---

## src/auth.py
### TODO 1: Add password hashing before storing credentials
- **Line:** 2
- **Context:**
  ```python
  # Authentication module
  # TODO: Add password hashing before storing credentials
  def login(user, password):
  ```
- **Details:** The authenticatio

## What the Agent Just Did — Autonomous Decisions

Let's step back and look at what the agent decided to do on its own:

| Agent decision | Why it matters |
|----------------|----------------|
| Called `AskUserQuestion` first | It understood preferences were needed before doing any work |
| Used `Glob` before `Grep` | Smart sequencing — map the codebase first, then search within it |
| Used `Grep` to find TODOs | Efficient — no need to `Read` every file; search first |
| Used `Read` only in Detailed mode | Respected the user's preference; no wasted tool calls |
| Grouped the report by file | Logical structure that makes the output easy to act on |

**None of those decisions were in your code.** You gave the agent a goal and a set of
allowed tools. It figured out the rest. That is autonomous tool orchestration.

## Section 3 Wrap-Up — Every Built-In Tool Covered

You have now completed **Section 3: Built-In Tools Deep Dive**.

| Lecture | Tools | Category |
|---------|-------|----------|
| 3.1 | `Read`, `Write`, `Edit` | File tools |
| 3.2 | `Bash`, `Monitor` | Shell tools |
| 3.3 | `Glob`, `Grep` | Search tools |
| 3.4 | `WebSearch`, `WebFetch` | Web tools |
| 3.5 | `AskUserQuestion` + `can_use_tool` | Human-in-the-loop |
| 3.6 | All of the above, working together | Real agent |

**You now know every built-in tool the Claude Agent SDK provides.**

---

### What's Next — Section 4: Permissions & Safety

In Section 4 you will learn how to control exactly what your agents are **allowed** to do:
- How `allowed_tools` works under the hood
- Read-only agents vs. auto-edit agents
- How to block dangerous operations before they execute
- Building a safe code review agent that can analyse code but cannot modify it